# m34 lab · A mapping-system data pipeline

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

This lab builds a small version of the data path in module m34 and measures it.

| Part | What you do | Data |
| --- | --- | --- |
| A | Generate an 8-channel stream from the m03 electrogram model, quantize it, and measure its data rate | synthetic |
| B | Push it through a bounded buffer with a consumer that stalls; find the buffer that loses nothing | synthetic |
| C | Check Little's law and the single-server queue formula by simulation | synthetic |
| D | Align a location stream with the electrogram stream; measure and then remove the misregistration | synthetic |
| E | Store a small session in an OpenEP-style structure and in WFDB format; read both back | synthetic |

**How to run.** From the course repository: `pip install -r requirements.txt`, then open this notebook in Jupyter.
In Google Colab the first code cell fetches the course code for you. The whole lab runs in under a minute.

Cells that start with `assert` are **checks**: they pass with the code as given. If you change something and a
check fails, that is information, not an error to hide. All data here are synthetic or come from the course's own
models; nothing describes a real product or patient.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Find the shared model code in labs/python/epml. In Colab, fetch the course repository first.
try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/open-ep-mapping-lab/open-ep-mapping-lab.github.io",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

import tempfile
import wfdb

from epml import electrograms as eg
from epml import systems as sy

## Part A · A stream and its data rate

Eight channels sampled at 1 kHz with 14-bit resolution, the layout of each record in the Intracardiac Atrial
Fibrillation Database used in m03. Five channels are bipoles 7 mm apart along a catheter, seen by the m03 model as a
planar front passes at 1 mm/ms; three are slow placeholder channels. The beat interval is an illustrative input.

The module's data rate is $R = N f_s b$. Samples with 14-bit resolution are usually stored in 16-bit words, so the
stored rate and the information rate differ.

In [ ]:
fs, n_ch, bits = 1000.0, 8, 14
t = np.arange(0, 2000.0, 1000 / fs)            # ms
positions = np.arange(5) * 7.0                  # mm along the catheter
beats = np.arange(100.0, 2000.0, 400.0)         # ms, illustrative
sig = np.zeros((t.size, n_ch))
for k, x in enumerate(positions):
    for tb in beats:
        sig[:, k] += eg.bipolar_planar(x - 1.0 * (t - tb), 2.0, 0.0, 1.3)   # front at 1 mm/ms
for k in range(5, 8):
    sig[:, k] = 0.2 * np.sin(2 * np.pi * (k - 4) * t / 1000)

scale = (2 ** (bits - 1) - 1) / np.abs(sig).max()   # use the full 14-bit range
q = np.round(sig * scale).astype(np.int16)          # stored in 16-bit words

one_second = q[: int(fs)]
stored_bps = one_second.nbytes * 8
print(f"stored {stored_bps / 1e3:.0f} kbit/s, information {sy.data_rate_bps(n_ch, fs, bits) / 1e3:.0f} kbit/s")

fig, ax = plt.subplots()
for k in range(5):
    ax.plot(t, sig[:, k] - 0.4 * k, lw=1)
ax.set(xlabel="time (ms)", ylabel="bipoles (offset)", title="Five bipoles as the front passes")
plt.show()

In [ ]:
assert stored_bps == sy.data_rate_bps(n_ch, fs, 16)
assert sy.data_rate_bps(n_ch, fs, bits) / stored_bps == 14 / 16
assert np.max(np.abs(q / scale - sig)) <= 0.5 / scale + 1e-12   # quantization error is at most half a step

## Part B · A bounded buffer and a stalling consumer

Blocks arrive one per time step. The consumer can handle `service` blocks per step on average, a little more than
arrives, but stops completely for `stall` steps once. Blocks wait in a ring buffer; a full buffer drops the newest
block. The module says a buffer of $R\,T$ rides out a stall of length $T$: here that is `stall` blocks, plus the one
block that is normally waiting.

In [ ]:
def run_pipeline(n_steps, service, stall_at, stall, capacity):
    """Latency (steps) of every block that got through, and the number dropped."""
    buf, credit, latency = sy.RingBuffer(capacity), 0.0, []
    for k in range(n_steps):
        if not (stall_at <= k < stall_at + stall):
            credit = min(credit + service, service + 1.0)
            while credit >= 1 and len(buf):
                latency.append(k - buf.pop())
                credit -= 1
        buf.push(k)
    return np.array(latency), buf.dropped

stall, service = 40, 1.25
for capacity in (stall, stall + 1, 2 * stall):
    lat, dropped = run_pipeline(2000, service, 500, stall, capacity)
    print(f"capacity {capacity:3d}: dropped {dropped:2d}, worst latency {lat.max()} steps")

lat, dropped = run_pipeline(2000, service, 500, stall, stall + 1)
recovered = 500 + stall + np.argmax(lat[500 + stall:] <= 1)   # first block back at normal latency
fig, ax = plt.subplots()
ax.plot(lat)
ax.set(xlabel="block", ylabel="latency (steps)", title="A stall, then the queue drains at (service - 1) blocks per step")
plt.show()

In [ ]:
assert run_pipeline(2000, service, 500, stall, stall)[1] > 0        # one block short: data are lost
assert run_pipeline(2000, service, 500, stall, stall + 1)[1] == 0    # R T plus the waiting block: nothing lost
# Spare capacity sets the recovery time: the backlog drains at (service - 1) blocks per step.
assert abs((recovered - (500 + stall)) - stall / (service - 1)) <= 0.1 * stall / (service - 1)

## Part C · Little's law and the single-server queue

Random arrivals at rate $\lambda$ and random service at rate $\mu$: the module derives $W = 1/(\mu - \lambda)$ and
Little's law $L = \lambda W$. Here a simulation checks both, and the plot shows how fast latency grows near full load.

In [ ]:
rng = np.random.default_rng(34)
mu = 100.0                                   # items per second
loads = np.array([0.2, 0.4, 0.6, 0.8, 0.9])
W_sim = []
for rho in loads:
    arrivals, services = sy.poisson_queue(rho * mu, mu, 200_000, rng)
    _, depart = sy.simulate_queue(arrivals, services)
    W_sim.append(np.mean((depart - arrivals)[10_000:]))
W_sim = np.array(W_sim)
W_formula = np.array([sy.mm1_mean_time(r * mu, mu) for r in loads])

arrivals, services = sy.poisson_queue(0.7 * mu, mu, 50_000, rng)
_, depart = sy.simulate_queue(arrivals, services)
T = depart.max()
L = sy.time_average_in_system(arrivals, depart, 0.0, T)
lam_hat, W_hat = len(arrivals) / T, np.mean(depart - arrivals)

fig, ax = plt.subplots()
ax.plot(loads, 1000 * W_formula, "--", label="1/(mu - lambda)")
ax.plot(loads, 1000 * W_sim, "o", label="simulation")
ax.set(xlabel="load rho", ylabel="mean time in stage (ms)", title="Latency near full load")
ax.legend()
plt.show()

In [ ]:
assert np.allclose(W_sim, W_formula, rtol=0.08)
assert L == np.float64(L) and abs(L - lam_hat * W_hat) < 1e-9 * L   # exact on a sample path that starts and ends empty
assert W_formula[-1] / W_formula[0] > 7                             # 90% load vs 20% load

## Part D · Aligning two streams

A catheter tip moves on a circle of radius 10 mm at 20 mm/s (illustrative values). Its location is sampled every
50 ms; electrogram events carry their own timestamps. If the location stream is read 30 ms off, each point moves by
about $v\,\Delta t$; linear interpolation adds at most $h^2 \max\lvert\ddot x\rvert/8$ with
$\lvert\ddot x\rvert = v^2/r$. Correcting the offset with timestamps on a common clock leaves only the
interpolation error.

In [ ]:
r, v, h, offset = 10.0, 20.0, 0.050, 0.030      # mm, mm/s, s, s
w = v / r
t_loc = np.arange(0, 4.0, h)                    # location sample times (s)
x_loc, y_loc = r * np.cos(w * t_loc), r * np.sin(w * t_loc)
t_egm = np.arange(0.2, 3.5, 0.137)              # electrogram event times (s)

def located(times):
    return np.c_[np.interp(times, t_loc, x_loc), np.interp(times, t_loc, y_loc)]

truth = np.c_[r * np.cos(w * t_egm), r * np.sin(w * t_egm)]
err_offset = np.linalg.norm(located(t_egm + offset) - truth, axis=1)
err_fixed = np.linalg.norm(located(t_egm) - truth, axis=1)
bound_interp = sy.linear_interp_error_bound(h, v**2 / r)
print(f"offset {offset*1e3:.0f} ms: worst {err_offset.max():.2f} mm; corrected: worst {err_fixed.max():.3f} mm "
      f"(interpolation bound {bound_interp:.3f} mm)")

In [ ]:
assert err_offset.max() <= sy.misregistration(v, offset) + bound_interp + 1e-9
assert err_offset.max() > 0.9 * sy.misregistration(v, offset)
assert err_fixed.max() <= bound_interp + 1e-9

## Part E · Storing the session

Two formats. First an *OpenEP-style* structure: geometry, points with positions and annotations, electrograms with
their sampling rate and units, and the software version, saved with NumPy. It mirrors the kinds of content OpenEP
stores; it is not the OpenEP format itself. Second, the raw signals in PhysioNet's WFDB format, with the sampling
rate, gain and units in a header file.

In [ ]:
session = {
    "geometry_vertices": np.array([[0, 0, 0], [10, 0, 0], [0, 10, 0], [0, 0, 10]], float),   # mm
    "geometry_faces": np.array([[0, 1, 2], [0, 1, 3], [0, 2, 3], [1, 2, 3]]),
    "points_xyz": np.c_[truth, np.zeros(len(truth))],                                         # mm
    "points_time_s": t_egm,
    "egm_counts": q,
    "egm_fs_hz": np.float64(fs),
    "egm_gain_counts_per_unit": np.float64(scale),
    "egm_units": np.array("relative"),
    "software_version": np.array("m34-lab 1.0"),
}
tmp = Path(tempfile.mkdtemp())
np.savez_compressed(tmp / "session.npz", **session)
back = dict(np.load(tmp / "session.npz"))

wfdb.wrsamp("m34_session", fs=fs, units=["rel"] * n_ch, sig_name=[f"ch{k}" for k in range(n_ch)],
            d_signal=q.astype(np.int64), fmt=["16"] * n_ch, adc_gain=[float(scale)] * n_ch,
            baseline=[0] * n_ch, write_dir=str(tmp))
rec = wfdb.rdrecord(str(tmp / "m34_session"), physical=False)
print(rec.fs, rec.n_sig, rec.sig_len, (tmp / "m34_session.dat").stat().st_size, "bytes")

In [ ]:
assert all(np.array_equal(back[k], np.asarray(v)) for k, v in session.items())
assert rec.fs == fs and rec.n_sig == n_ch
assert np.array_equal(rec.d_signal, q)
assert (tmp / "m34_session.dat").stat().st_size == q.size * 2      # 16 bits per stored sample